<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 40%, #0d2137 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(88,166,255,0.1); border: 1px solid rgba(88,166,255,0.3); color: #58a6ff; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 6 (Java)</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Java Kafka Producer</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Build and run a custom Java producer to stream CSV data into Kafka.</p>
</div>

---

## 🎯 Overview

Java is the native language of the Kafka ecosystem. In this lab, you will run a custom Java application (`Producer.java`) that reads e-commerce data from `orders.csv` and publishes it to Kafka using the official `kafka-clients` library.

Because compiling Java typically requires setting up a build tool like Maven or Gradle on your machine, we are going to use a clever trick: we will copy our Java code directly into the running Kafka container and compile/run it there using the container's built-in Java compiler and Kafka libraries!

---

## ⚙️ Prerequisites

<div style="background-color: rgba(243, 156, 18, 0.1); border-left: 4px solid #f39c12; padding: 10px 15px; margin: 15px 0; border-radius: 4px;">
  <strong>⚠️ Important:</strong> Ensure any previous multi-broker clusters are stopped. Run <code>docker-compose down</code> in Lab5 before proceeding.
</div>

---

## <span style="color: #f85032;">Step 1:</span> Start the Kafka Cluster


In [ ]:
!docker-compose -f ../../docker-compose.yml up -d

---

## <span style="color: #f85032;">Step 2:</span> Create the Topic

Let's create the `orders` topic where our Java script will produce the data.

In [ ]:
!docker exec kafka kafka-topics \
  --bootstrap-server localhost:9092 \
  --create \
  --topic orders \
  --partitions 1 \
  --replication-factor 1

---

## <span style="color: #f85032;">Step 3:</span> Review `Producer.java`

Open `Producer.java` in your editor. Notice how it maps exactly to the Python version:
1. It configures `KafkaProducer` with `StringSerializer.class` for both Key and Value.
2. It uses a `BufferedReader` to read `orders.csv` line by line.
3. It manually formats the headers and values into a JSON-like string.
4. It calls `producer.send()` and passes an asynchronous `Callback` to print the delivery report.
5. It calls `producer.flush()` to ensure the network buffer is empty before quitting.

---
Let's copy the code and our CSV data into the Kafka container's `/tmp` folder.

In [ ]:
!docker cp Producer.java kafka:/tmp/Producer.java
!docker cp orders.csv kafka:/tmp/orders.csv

---

## <span style="color: #f85032;">Step 4:</span> Compile the Java Code

We will execute the `javac` compiler inside the container. We tell the compiler to look in `/usr/share/java/kafka/*` where the official Confluent `kafka-clients` JARs are located.

In [ ]:
!docker exec kafka bash -c "cd /tmp && javac -cp '/usr/share/java/kafka/*' Producer.java"

---

## <span style="color: #f85032;">Step 5:</span> Run the Java Producer

Now we run the compiled Java program! 
*(Note: You will see a harmless Log4j warning at the top because we didn't provide a logger config file, but the delivery reports will print successfully!)* 

In [ ]:
!docker exec kafka bash -c "cd /tmp && java -cp '.:/usr/share/java/kafka/*' Producer"

---

## <span style="color: #f85032;">Step 6:</span> Verify Data with Console Consumer

Let's verify the data actually made it into Kafka exactly like the Python version did.

In [ ]:
!docker exec kafka kafka-console-consumer \
  --bootstrap-server localhost:9092 \
  --topic orders \
  --from-beginning \
  --property print.key=true \
  --property key.separator=" - " \
  --timeout-ms 5000

---

## 🧹 Clean Up

Stop the cluster and remove volumes to clean up the disk space.

In [ ]:
!docker-compose -f ../../docker-compose.yml down -v

<div style="background-color: rgba(248, 80, 50, 0.1); border: 1px solid rgba(248, 80, 50, 0.3); padding: 20px; text-align: center; border-radius: 8px; margin-top: 40px;">
  <h3 style="color: #f85032; margin-bottom: 10px;">🎉 Lab 6 (Java) Complete!</h3>
  <p style="color: #8b949e; margin: 0;">You've successfully compiled and run a native Java Kafka Producer without installing Maven, reading from a CSV file and sending records with delivery callbacks!</p>
</div>